In [1]:
# ============================================================
# 0) IMPORTS
# ============================================================
from pathlib import Path
import os
import time
import numpy as np
import pandas as pd
import nibabel as nib
import scipy.ndimage as ndi
import torch
import torch.nn as nn


# ============================================================
# 1) PATHS
# KENDİ BİLGİSAYARINDAKİ YOLA GÖRE GEREKİRSE DÜZENLE
# ============================================================
PROJECT_ROOT = Path(r"C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segmentation")

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

MELA_DIR = DATA_DIR / "mela"
NSCLC_DIR = DATA_DIR / "nsclc"

ANNOT_DIR = MELA_DIR / "annotations"
IMG_DIR = MELA_DIR / "images"

NSCLC_MODEL_DIR = NSCLC_DIR / "models"
MODEL_PATH = NSCLC_MODEL_DIR / "best_model.pt"

PRED_SAVE_DIR = RESULTS_DIR / "mela_batch_predictions"
PRED_SAVE_DIR.mkdir(parents=True, exist_ok=True)

CSV_SAVE_PATH = RESULTS_DIR / "mela_batch_inference_summary.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ANNOT_DIR:", ANNOT_DIR)
print("IMG_DIR:", IMG_DIR)
print("MODEL_PATH:", MODEL_PATH)
print("MODEL EXISTS:", MODEL_PATH.exists())
print("PRED_SAVE_DIR:", PRED_SAVE_DIR)
print("CSV_SAVE_PATH:", CSV_SAVE_PATH)
print("DEVICE:", device)

PROJECT_ROOT: C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segmentation
ANNOT_DIR: C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segmentation\data\mela\annotations
IMG_DIR: C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segmentation\data\mela\images
MODEL_PATH: C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segmentation\data\nsclc\models\best_model.pt
MODEL EXISTS: True
PRED_SAVE_DIR: C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segmentation\results\mela_batch_predictions
CSV_SAVE_PATH: C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segmentation\results\mela_batch_inference_summary.csv
DEVICE: cpu


In [2]:
# ============================================================
# 2) MODEL CLASS
# ============================================================
class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()

        def CBR(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True)
            )

        self.enc1 = nn.Sequential(CBR(1, 32), CBR(32, 32))
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = nn.Sequential(CBR(32, 64), CBR(64, 64))
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = nn.Sequential(CBR(64, 128), CBR(128, 128))
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = nn.Sequential(CBR(128, 256), CBR(256, 256))

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(CBR(256, 128), CBR(128, 128))

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(CBR(128, 64), CBR(64, 64))

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(CBR(64, 32), CBR(32, 32))

        self.out = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        e3 = self.enc3(p2)
        p3 = self.pool3(e3)

        b = self.bottleneck(p3)

        u3 = self.up3(b)
        d3 = self.dec3(torch.cat([u3, e3], dim=1))

        u2 = self.up2(d3)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        out = self.out(d1)
        return out

In [3]:
# ============================================================
# 2) MODEL CLASS
# EĞİTİMDE KULLANILAN CLASS İLE AYNI OLMALI
# ============================================================
class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()

        def CBR(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True)
            )

        self.enc1 = nn.Sequential(CBR(1, 32), CBR(32, 32))
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = nn.Sequential(CBR(32, 64), CBR(64, 64))
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = nn.Sequential(CBR(64, 128), CBR(128, 128))
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = nn.Sequential(CBR(128, 256), CBR(256, 256))

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(CBR(256, 128), CBR(128, 128))

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(CBR(128, 64), CBR(64, 64))

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(CBR(64, 32), CBR(32, 32))

        self.out = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        e3 = self.enc3(p2)
        p3 = self.pool3(e3)

        b = self.bottleneck(p3)

        u3 = self.up3(b)
        d3 = self.dec3(torch.cat([u3, e3], dim=1))

        u2 = self.up2(d3)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        out = self.out(d1)
        return out

In [4]:
# ============================================================
# 3) LOAD MODEL
# ============================================================
model = SimpleUNet().to(device)
state_dict = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print("✅ Model loaded successfully")

✅ Model loaded successfully


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_6928\1906258660.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(MODEL_PATH, map_location=device)


In [5]:
# ============================================================
# 4) LOAD MELA TABLES
# ============================================================
ann_df = pd.read_csv(ANNOT_DIR / "mela_train_val_annotations.csv")
spacing_df = pd.read_csv(ANNOT_DIR / "mela_origin_spacing.csv")

print("Annotations shape:", ann_df.shape)
print("Spacing shape:", spacing_df.shape)

df = ann_df.merge(spacing_df, on="public_id", how="left")
print("Merged dataframe shape:", df.shape)

display(df.head())

Annotations shape: (884, 7)
Spacing shape: (880, 7)
Merged dataframe shape: (884, 13)


,public_id,coordX,coordY,coordZ,x_length,y_length,z_length,origin_x,origin_y,origin_z,spacing_x,spacing_y,spacing_z
0,mela_0001,300,153,278,62,61,37,-162.130005,15.700000,-394.970001,0.605469,0.605469,1.000000
1,mela_0002,265,182,274,42,39,38,-181.210083,-185.636719,-507.846436,0.726562,0.726562,1.000000
2,mela_0003,259,269,315,111,81,100,-157.666016,-357.166016,-226.100006,0.667969,0.667969,0.700012
3,mela_0004,317,309,360,112,115,138,-199.605469,-351.605469,-376.899994,0.789062,0.789062,0.699982
4,mela_0005,284,187,345,40,59,54,-179.621094,-343.621094,167.600006,0.757812,0.757812,0.699997


In [6]:
# ============================================================
# 5) HELPERS
# ============================================================
def find_image_path(public_id, img_root):
    img_root = Path(img_root)

    candidate_patterns = [
        img_root / "train" / f"{public_id}.nii.gz",
        img_root / "val" / f"{public_id}.nii.gz",
        img_root / f"{public_id}.nii.gz",
        img_root / "train" / f"{public_id}.nii",
        img_root / "val" / f"{public_id}.nii",
        img_root / f"{public_id}.nii",
    ]

    for p in candidate_patterns:
        if p.exists():
            if "train" in str(p):
                split_name = "train"
            elif "val" in str(p):
                split_name = "val"
            else:
                split_name = "unknown"
            return p, split_name

    matches = list(img_root.rglob(f"{public_id}*"))
    if len(matches) > 0:
        p = matches[0]
        if "train" in str(p):
            split_name = "train"
        elif "val" in str(p):
            split_name = "val"
        else:
            split_name = "unknown"
        return p, split_name

    return None, None


def pad_to_multiple_2d(img_2d, multiple=8):
    h, w = img_2d.shape

    new_h = int(np.ceil(h / multiple) * multiple)
    new_w = int(np.ceil(w / multiple) * multiple)

    pad_h = new_h - h
    pad_w = new_w - w

    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left

    padded = np.pad(
        img_2d,
        ((pad_top, pad_bottom), (pad_left, pad_right)),
        mode="constant",
        constant_values=0
    )

    pad_info = {
        "pad_top": pad_top,
        "pad_bottom": pad_bottom,
        "pad_left": pad_left,
        "pad_right": pad_right,
        "orig_h": h,
        "orig_w": w,
    }
    return padded, pad_info


def unpad_2d(img_2d, pad_info):
    pt = pad_info["pad_top"]
    pb = pad_info["pad_bottom"]
    pl = pad_info["pad_left"]
    pr = pad_info["pad_right"]

    h_end = img_2d.shape[0] - pb if pb > 0 else img_2d.shape[0]
    w_end = img_2d.shape[1] - pr if pr > 0 else img_2d.shape[1]

    return img_2d[pt:h_end, pl:w_end]


def largest_component_2d(mask_2d):
    labeled, num = ndi.label(mask_2d)
    if num == 0:
        return mask_2d.astype(np.uint8)

    sizes = ndi.sum(mask_2d, labeled, range(1, num + 1))
    max_label = np.argmax(sizes) + 1
    return (labeled == max_label).astype(np.uint8)


def largest_component_3d(mask_3d):
    labeled, num = ndi.label(mask_3d)
    if num == 0:
        return mask_3d.astype(np.uint8)

    sizes = ndi.sum(mask_3d, labeled, range(1, num + 1))
    max_label = np.argmax(sizes) + 1
    return (labeled == max_label).astype(np.uint8)


def make_bbox_mask(volume_shape, x1, x2, y1, y2, z1, z2):
    bbox_mask = np.zeros(volume_shape, dtype=np.uint8)
    bbox_mask[z1:z2, y1:y2, x1:x2] = 1
    return bbox_mask

In [7]:
# ============================================================
# 6) SINGLE CASE FUNCTION
# BATCH İÇİN BUNU ÇAĞIRACAĞIZ
# ============================================================
def run_mela_single_case(
    sample_row,
    model,
    img_root,
    threshold=0.80,
    z_margin=3,
    roi_margin=10,
    apply_2d_lcc=True,
    apply_3d_lcc=True,
    apply_bbox_constraint=True,
):
    public_id = sample_row["public_id"]

    img_path, split_name = find_image_path(public_id, img_root)
    if img_path is None:
        return {
            "public_id": public_id,
            "status": "fail",
            "reason": "image_not_found"
        }

    try:
        nii = nib.load(str(img_path))
        vol_xyz = nii.get_fdata()
        volume = np.transpose(vol_xyz, (2, 1, 0)).astype(np.float32)
    except Exception as e:
        return {
            "public_id": public_id,
            "status": "fail",
            "reason": f"load_error: {str(e)}"
        }

    try:
        cx = int(round(sample_row["coordX"]))
        cy = int(round(sample_row["coordY"]))
        cz = int(round(sample_row["coordZ"]))

        lx = int(round(sample_row["x_length"]))
        ly = int(round(sample_row["y_length"]))
        lz = int(round(sample_row["z_length"]))
    except Exception as e:
        return {
            "public_id": public_id,
            "status": "fail",
            "reason": f"bbox_parse_error: {str(e)}"
        }

    x1 = max(0, cx - lx // 2 - roi_margin)
    x2 = min(volume.shape[2], cx + lx // 2 + roi_margin)

    y1 = max(0, cy - ly // 2 - roi_margin)
    y2 = min(volume.shape[1], cy + ly // 2 + roi_margin)

    z1 = max(0, cz - lz // 2 - roi_margin)
    z2 = min(volume.shape[0], cz + lz // 2 + roi_margin)

    roi = volume[z1:z2, y1:y2, x1:x2]

    if roi.size == 0:
        return {
            "public_id": public_id,
            "status": "fail",
            "reason": "empty_roi"
        }

    roi_norm = np.clip(roi, -1000, 400)
    roi_norm = (roi_norm + 1000.0) / 1400.0
    roi_norm = roi_norm.astype(np.float32)

    pred_slices = []

    try:
        with torch.no_grad():
            for z_idx in range(roi_norm.shape[0]):
                img_2d = roi_norm[z_idx]

                padded, pad_info = pad_to_multiple_2d(img_2d, multiple=8)
                x = torch.from_numpy(padded).unsqueeze(0).unsqueeze(0).to(device)

                logits = model(x)
                prob = torch.sigmoid(logits).squeeze().cpu().numpy()
                prob = unpad_2d(prob, pad_info)

                pred = (prob >= threshold).astype(np.uint8)

                if apply_2d_lcc and pred.sum() > 0:
                    pred = largest_component_2d(pred)

                pred_slices.append(pred)

        pred_roi = np.stack(pred_slices, axis=0).astype(np.uint8)

    except Exception as e:
        return {
            "public_id": public_id,
            "status": "fail",
            "reason": f"inference_error: {str(e)}"
        }

    if apply_3d_lcc and pred_roi.sum() > 0:
        pred_roi = largest_component_3d(pred_roi)

    full_pred = np.zeros_like(volume, dtype=np.uint8)
    full_pred[z1:z2, y1:y2, x1:x2] = pred_roi

    z_half = lz // 2
    valid_z1 = max(0, cz - z_half - z_margin)
    valid_z2 = min(volume.shape[0], cz + z_half + z_margin)

    full_pred[:valid_z1] = 0
    full_pred[valid_z2:] = 0

    if apply_bbox_constraint:
        bbox_mask = make_bbox_mask(volume.shape, x1, x2, y1, y2, z1, z2)
        full_pred = full_pred * bbox_mask

    if apply_3d_lcc and full_pred.sum() > 0:
        full_pred = largest_component_3d(full_pred)

    nonzero_slices = np.where(full_pred.reshape(full_pred.shape[0], -1).sum(axis=1) > 0)[0]

    result = {
        "public_id": public_id,
        "status": "ok",
        "reason": "",

        "split": split_name,
        "img_path": str(img_path),

        "volume_shape_z": int(volume.shape[0]),
        "volume_shape_y": int(volume.shape[1]),
        "volume_shape_x": int(volume.shape[2]),

        "cx": int(cx),
        "cy": int(cy),
        "cz": int(cz),

        "lx": int(lx),
        "ly": int(ly),
        "lz": int(lz),

        "x1": int(x1),
        "x2": int(x2),
        "y1": int(y1),
        "y2": int(y2),
        "z1": int(z1),
        "z2": int(z2),

        "roi_shape_z": int(roi.shape[0]),
        "roi_shape_y": int(roi.shape[1]),
        "roi_shape_x": int(roi.shape[2]),

        "threshold": float(threshold),
        "z_margin": int(z_margin),
        "roi_margin": int(roi_margin),

        "pred_sum": int(full_pred.sum()),
        "roi_pred_sum": int(pred_roi.sum()),
        "nonzero_slice_count": int(len(nonzero_slices)),
        "first_nonzero_slice": int(nonzero_slices[0]) if len(nonzero_slices) > 0 else -1,
        "last_nonzero_slice": int(nonzero_slices[-1]) if len(nonzero_slices) > 0 else -1,
        "empty_prediction": int(full_pred.sum() == 0),

        "bbox_volume_proxy": int(max(lx, 0) * max(ly, 0) * max(lz, 0)),
        "pred_path": "",
        "full_pred": full_pred,
    }

    return result

In [8]:
# ============================================================
# 7) QUICK SINGLE TEST
# İSTEĞE BAĞLI: BATCH ÖNCESİ TEK VAKA TESTİ
# ============================================================
sample_row = df.iloc[0].copy()

test_result = run_mela_single_case(
    sample_row=sample_row,
    model=model,
    img_root=IMG_DIR,
    threshold=0.80,
    z_margin=3,
    roi_margin=10,
    apply_2d_lcc=True,
    apply_3d_lcc=True,
    apply_bbox_constraint=True,
)

print("Test public_id:", test_result["public_id"])
print("Test status:", test_result["status"])
print("Reason:", test_result.get("reason", ""))
if test_result["status"] == "ok":
    print("pred_sum:", test_result["pred_sum"])
    print("nonzero_slice_count:", test_result["nonzero_slice_count"])

Test public_id: mela_0001
Test status: ok
Reason: 
pred_sum: 11271
nonzero_slice_count: 31


In [9]:
# ============================================================
# 8) BATCH INFERENCE SETTINGS
# İSTERSEN ÖNCE KÜÇÜK SUBSETTE DENE
# ============================================================

RUN_ONLY_FIRST_N = None   # örn: 20 yazarsan ilk 20 vaka üzerinde çalışır
SAVE_PREDICTIONS = True

THRESHOLD = 0.80
Z_MARGIN = 3
ROI_MARGIN = 10
APPLY_2D_LCC = True
APPLY_3D_LCC = True
APPLY_BBOX_CONSTRAINT = True

if RUN_ONLY_FIRST_N is not None:
    run_df = df.iloc[:RUN_ONLY_FIRST_N].copy()
else:
    run_df = df.copy()

print("Run sample count:", len(run_df))

Run sample count: 884


In [14]:
# ============================================================
# BATCH INFERENCE SETTINGS
# İSTERSEN ÖNCE KÜÇÜK SUBSETTE DENE
# ============================================================

RUN_ONLY_FIRST_N = 20   # önce test için 20 yap
SAVE_PREDICTIONS = True

THRESHOLD = 0.80
Z_MARGIN = 3
ROI_MARGIN = 10
APPLY_2D_LCC = True
APPLY_3D_LCC = True
APPLY_BBOX_CONSTRAINT = True

if RUN_ONLY_FIRST_N is not None:
    run_df = valid_df.iloc[:RUN_ONLY_FIRST_N].copy()
else:
    run_df = valid_df.copy()

print("Run sample count:", len(run_df))

NameError: name 'valid_df' is not defined

In [15]:
# ============================================================
# VALID SAMPLE FILTER
# Sadece image dosyası gerçekten bulunan vakaları tut
# ============================================================
valid_rows = []

for _, row in df.iterrows():
    public_id = row["public_id"]
    img_path, split_name = find_image_path(public_id, IMG_DIR)
    if img_path is not None:
        valid_rows.append(row.to_dict())

valid_df = pd.DataFrame(valid_rows).reset_index(drop=True)

print("Original merged df count:", len(df))
print("Valid df count (existing images only):", len(valid_df))

display(valid_df.head())

Original merged df count: 884
Valid df count (existing images only): 372


,public_id,coordX,coordY,coordZ,x_length,y_length,z_length,origin_x,origin_y,origin_z,spacing_x,spacing_y,spacing_z
0,mela_0001,300,153,278,62,61,37,-162.130005,15.700000,-394.970001,0.605469,0.605469,1.000000
1,mela_0002,265,182,274,42,39,38,-181.210083,-185.636719,-507.846436,0.726562,0.726562,1.000000
2,mela_0003,259,269,315,111,81,100,-157.666016,-357.166016,-226.100006,0.667969,0.667969,0.700012
3,mela_0004,317,309,360,112,115,138,-199.605469,-351.605469,-376.899994,0.789062,0.789062,0.699982
4,mela_0005,284,187,345,40,59,54,-179.621094,-343.621094,167.600006,0.757812,0.757812,0.699997


In [18]:
# ============================================================
# BATCH INFERENCE SETTINGS
# İSTERSEN ÖNCE KÜÇÜK SUBSETTE DENE
# ============================================================

RUN_ONLY_FIRST_N = None
SAVE_PREDICTIONS = True

THRESHOLD = 0.80
Z_MARGIN = 3
ROI_MARGIN = 10
APPLY_2D_LCC = True
APPLY_3D_LCC = True
APPLY_BBOX_CONSTRAINT = True

if RUN_ONLY_FIRST_N is not None:
    run_df = valid_df.iloc[:RUN_ONLY_FIRST_N].copy()
else:
    run_df = valid_df.copy()

print("Run sample count:", len(run_df))

Run sample count: 372


In [19]:
# ============================================================
# BATCH INFERENCE
# ============================================================
results = []

start_time = time.time()

for idx, (_, row) in enumerate(run_df.iterrows(), start=1):
    public_id = row["public_id"]

    result = run_mela_single_case(
        sample_row=row,
        model=model,
        img_root=IMG_DIR,
        threshold=THRESHOLD,
        z_margin=Z_MARGIN,
        roi_margin=ROI_MARGIN,
        apply_2d_lcc=APPLY_2D_LCC,
        apply_3d_lcc=APPLY_3D_LCC,
        apply_bbox_constraint=APPLY_BBOX_CONSTRAINT,
    )

    if result["status"] == "ok" and SAVE_PREDICTIONS:
        pred_path = PRED_SAVE_DIR / f"{public_id}_pred.npy"
        np.save(pred_path, result["full_pred"])
        result["pred_path"] = str(pred_path)

    if "full_pred" in result:
        del result["full_pred"]

    results.append(result)

    if idx % 10 == 0 or idx == len(run_df):
        ok_count = sum(r["status"] == "ok" for r in results)
        fail_count = sum(r["status"] != "ok" for r in results)
        print(f"[{idx}/{len(run_df)}] processed | ok={ok_count} | fail={fail_count}")

elapsed = time.time() - start_time
print(f"\nBatch inference completed in {elapsed:.2f} seconds")

[10/372] processed | ok=10 | fail=0
[20/372] processed | ok=20 | fail=0
[30/372] processed | ok=30 | fail=0
[40/372] processed | ok=40 | fail=0
[50/372] processed | ok=50 | fail=0
[60/372] processed | ok=60 | fail=0
[70/372] processed | ok=70 | fail=0
[80/372] processed | ok=80 | fail=0
[90/372] processed | ok=90 | fail=0
[100/372] processed | ok=100 | fail=0
[110/372] processed | ok=110 | fail=0
[120/372] processed | ok=120 | fail=0
[130/372] processed | ok=130 | fail=0
[140/372] processed | ok=140 | fail=0
[150/372] processed | ok=150 | fail=0
[160/372] processed | ok=160 | fail=0
[170/372] processed | ok=170 | fail=0
[180/372] processed | ok=180 | fail=0
[190/372] processed | ok=190 | fail=0
[200/372] processed | ok=200 | fail=0
[210/372] processed | ok=210 | fail=0
[220/372] processed | ok=220 | fail=0
[230/372] processed | ok=230 | fail=0
[240/372] processed | ok=240 | fail=0
[250/372] processed | ok=250 | fail=0
[260/372] processed | ok=260 | fail=0
[270/372] processed | ok=270 |

In [20]:
# ============================================================
# RESULTS DATAFRAME
# ============================================================
results_df = pd.DataFrame(results)

print("Results dataframe shape:", results_df.shape)
display(results_df.head())

print("\nStatus counts:")
print(results_df["status"].value_counts(dropna=False))

if "reason" in results_df.columns:
    print("\nFailure reasons:")
    print(results_df["reason"].value_counts(dropna=False))

if "empty_prediction" in results_df.columns:
    print("\nEmpty prediction counts:")
    print(results_df["empty_prediction"].value_counts(dropna=False))

Results dataframe shape: (372, 34)


,public_id,status,reason,split,img_path,volume_shape_z,volume_shape_y,volume_shape_x,cx,cy,...,z_margin,roi_margin,pred_sum,roi_pred_sum,nonzero_slice_count,first_nonzero_slice,last_nonzero_slice,empty_prediction,bbox_volume_proxy,pred_path
0,mela_0001,ok,,train,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...,385,512,512,300,153,...,3,10,11271,17270,31,257,287,0,139934,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
1,mela_0002,ok,,train,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...,376,512,512,265,182,...,3,10,0,30,0,-1,-1,1,62244,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
2,mela_0003,ok,,train,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...,429,512,512,259,269,...,3,10,12471,12471,19,332,350,0,899100,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
3,mela_0004,ok,,train,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...,453,512,512,317,309,...,3,10,422174,422174,135,293,427,0,1777440,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
4,mela_0005,ok,,train,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...,453,512,512,284,187,...,3,10,284,284,5,364,368,0,127440,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...



Status counts:
status
ok    372
Name: count, dtype: int64

Failure reasons:
reason
    372
Name: count, dtype: int64

Empty prediction counts:
empty_prediction
0    324
1     48
Name: count, dtype: int64


In [21]:
# ============================================================
# SAVE CSV
# ============================================================
results_df.to_csv(CSV_SAVE_PATH, index=False)
print("✅ Saved summary CSV:", CSV_SAVE_PATH)

✅ Saved summary CSV: C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segmentation\results\mela_batch_inference_summary.csv


In [22]:
ok_df = results_df[results_df["status"] == "ok"].copy()

print("Total OK:", len(ok_df))
print("Empty predictions:", (ok_df["empty_prediction"] == 1).sum())
print("Non-empty predictions:", (ok_df["empty_prediction"] == 0).sum())

non_empty_df = ok_df[ok_df["empty_prediction"] == 0].copy()

print("\nPred sum stats (non-empty only):")
print(non_empty_df["pred_sum"].describe())

print("\nTop 10 largest predictions:")
display(non_empty_df.sort_values("pred_sum", ascending=False)[["public_id", "pred_sum", "nonzero_slice_count"]].head(10))

print("\nTop 10 smallest non-empty predictions:")
display(non_empty_df.sort_values("pred_sum", ascending=True)[["public_id", "pred_sum", "nonzero_slice_count"]].head(10))

Total OK: 372
Empty predictions: 48
Non-empty predictions: 324

Pred sum stats (non-empty only):
count    3.240000e+02
mean     3.752859e+04
std      1.119490e+05
min      2.000000e+00
25%      8.255000e+02
50%      5.437000e+03
75%      2.310100e+04
max      1.416263e+06
Name: pred_sum, dtype: float64

Top 10 largest predictions:


,public_id,pred_sum,nonzero_slice_count
361,mela_0870,1416263,246
10,mela_0011,643619,124
136,mela_0136,625768,121
143,mela_0143,466687,104
332,mela_0841,438758,141
3,mela_0004,422174,135
275,mela_0784,344519,100
19,mela_0020,315495,71
365,mela_0874,305312,121
28,mela_0029,301030,121



Top 10 smallest non-empty predictions:


,public_id,pred_sum,nonzero_slice_count
258,mela_0257,2,1
215,mela_0214,3,1
74,mela_0075,7,1
174,mela_0174,8,2
32,mela_0033,11,4
13,mela_0014,12,1
267,mela_0776,15,1
14,mela_0015,16,1
115,mela_0115,21,2
176,mela_0176,27,5


In [23]:
# ============================================================
# MELA PROXY SUMMARY
# Accuracy yerine kullanılabilecek yardımcı göstergeler
# ============================================================
ok_df = results_df[results_df["status"] == "ok"].copy()

total_ok = len(ok_df)
empty_count = int((ok_df["empty_prediction"] == 1).sum())
non_empty_count = int((ok_df["empty_prediction"] == 0).sum())

empty_rate = empty_count / total_ok if total_ok > 0 else 0
non_empty_rate = non_empty_count / total_ok if total_ok > 0 else 0

print("Total valid cases:", total_ok)
print("Empty predictions:", empty_count)
print("Non-empty predictions:", non_empty_count)
print(f"Empty prediction rate: {empty_rate:.4f} ({empty_rate*100:.2f}%)")
print(f"Non-empty prediction rate: {non_empty_rate:.4f} ({non_empty_rate*100:.2f}%)")

Total valid cases: 372
Empty predictions: 48
Non-empty predictions: 324
Empty prediction rate: 0.1290 (12.90%)
Non-empty prediction rate: 0.8710 (87.10%)


In [24]:
# ============================================================
# EXAMPLE CASE GROUPS
# 4 grup örnek vaka listesi çıkar
# ============================================================
ok_df = results_df[results_df["status"] == "ok"].copy()
non_empty_df = ok_df[ok_df["empty_prediction"] == 0].copy()
empty_df = ok_df[ok_df["empty_prediction"] == 1].copy()

# Large examples: en büyük tahminler
large_examples = non_empty_df.sort_values("pred_sum", ascending=False).head(5).copy()

# Small examples: en küçük non-empty tahminler
small_examples = non_empty_df.sort_values("pred_sum", ascending=True).head(5).copy()

# Empty examples: ilk 5 empty
empty_examples = empty_df.head(5).copy()

# Medium examples: median civarındaki örnekler
median_pred_sum = non_empty_df["pred_sum"].median()
non_empty_df["median_distance"] = (non_empty_df["pred_sum"] - median_pred_sum).abs()
medium_examples = non_empty_df.sort_values("median_distance", ascending=True).head(5).copy()

print("Large examples:")
display(large_examples[["public_id", "pred_sum", "nonzero_slice_count", "pred_path"]])

print("Small examples:")
display(small_examples[["public_id", "pred_sum", "nonzero_slice_count", "pred_path"]])

print("Empty examples:")
display(empty_examples[["public_id", "pred_sum", "nonzero_slice_count", "pred_path"]])

print("Medium examples:")
display(medium_examples[["public_id", "pred_sum", "nonzero_slice_count", "pred_path"]])

Large examples:


,public_id,pred_sum,nonzero_slice_count,pred_path
361,mela_0870,1416263,246,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
10,mela_0011,643619,124,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
136,mela_0136,625768,121,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
143,mela_0143,466687,104,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
332,mela_0841,438758,141,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...


Small examples:


,public_id,pred_sum,nonzero_slice_count,pred_path
258,mela_0257,2,1,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
215,mela_0214,3,1,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
74,mela_0075,7,1,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
174,mela_0174,8,2,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
32,mela_0033,11,4,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...


Empty examples:


,public_id,pred_sum,nonzero_slice_count,pred_path
1,mela_0002,0,0,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
7,mela_0008,0,0,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
9,mela_0010,0,0,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
18,mela_0019,0,0,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
21,mela_0022,0,0,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...


Medium examples:


,public_id,pred_sum,nonzero_slice_count,pred_path
318,mela_0827,5396,38,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
17,mela_0018,5478,18,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
75,mela_0076,5509,14,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
153,mela_0153,5549,18,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
224,mela_0223,5574,26,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...


In [25]:
# ============================================================
# SAVE SELECTED EXAMPLE CASES
# ============================================================
large_examples = large_examples.copy()
large_examples["example_group"] = "large"

small_examples = small_examples.copy()
small_examples["example_group"] = "small"

empty_examples = empty_examples.copy()
empty_examples["example_group"] = "empty"

medium_examples = medium_examples.copy()
medium_examples["example_group"] = "medium"

selected_examples_df = pd.concat(
    [large_examples, small_examples, empty_examples, medium_examples],
    axis=0,
    ignore_index=True
)

selected_examples_df = selected_examples_df.drop_duplicates(subset=["public_id"]).reset_index(drop=True)

EXAMPLE_SAVE_PATH = RESULTS_DIR / "mela_selected_example_cases.csv"
selected_examples_df.to_csv(EXAMPLE_SAVE_PATH, index=False)

print("✅ Saved example case list:", EXAMPLE_SAVE_PATH)
display(selected_examples_df[["public_id", "example_group", "pred_sum", "nonzero_slice_count", "pred_path"]])

✅ Saved example case list: C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segmentation\results\mela_selected_example_cases.csv


,public_id,example_group,pred_sum,nonzero_slice_count,pred_path
0,mela_0870,large,1416263,246,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
1,mela_0011,large,643619,124,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
2,mela_0136,large,625768,121,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
3,mela_0143,large,466687,104,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
4,mela_0841,large,438758,141,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
5,mela_0257,small,2,1,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
6,mela_0214,small,3,1,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
7,mela_0075,small,7,1,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
8,mela_0174,small,8,2,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...
9,mela_0033,small,11,4,C:\Users\LENOVO\Desktop\3D-Lung-Lesion-Segment...


In [26]:
# ============================================================
# QUICK LISTS
# ============================================================
print("Large public_ids:", large_examples["public_id"].tolist())
print("Small public_ids:", small_examples["public_id"].tolist())
print("Empty public_ids:", empty_examples["public_id"].tolist())
print("Medium public_ids:", medium_examples["public_id"].tolist())

Large public_ids: ['mela_0870', 'mela_0011', 'mela_0136', 'mela_0143', 'mela_0841']
Small public_ids: ['mela_0257', 'mela_0214', 'mela_0075', 'mela_0174', 'mela_0033']
Empty public_ids: ['mela_0002', 'mela_0008', 'mela_0010', 'mela_0019', 'mela_0022']
Medium public_ids: ['mela_0827', 'mela_0018', 'mela_0076', 'mela_0153', 'mela_0223']
